In [1]:
# %% Cell 0: thư viện + tham số
import os
import datetime
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

CSV = r"D:\parkinsons_v1.csv"
SEED = 42
K_OUTER = 10   # số fold (nhóm chưa chốt 10 hay 5)
K_INNER = 5    # CV bên trong để chọn tham số

# %% Bước 1: biến mục tiêu
df = pd.read_csv(CSV, sep=None, engine="python")   # tự nhận dấu phân cách (phẩy hoặc tab)
y = df["status"]
print(df.shape)
print(y.value_counts())

# %% Bước 2: X (22 feature) và y
X = df.drop(columns=["status"])
assert X.shape[1] == 22 and "name" not in X.columns
assert X.isna().sum().sum() == 0

# %% Bước 3-4: CV ngoài và CV trong
outer_cv = StratifiedKFold(n_splits=K_OUTER, shuffle=True, random_state=SEED)
inner_cv = StratifiedKFold(n_splits=K_INNER, shuffle=True, random_state=SEED)

# %% Bước 5-6: 3 mô hình + lưới tham số (scaler nằm trong Pipeline để không bị leakage)
models = {
    "KNN": (
        Pipeline([("scaler", StandardScaler()), ("clf", KNeighborsClassifier())]),
        {"clf__n_neighbors": [3, 5, 7]},
    ),
    "SVM": (
        Pipeline([("scaler", StandardScaler()), ("clf", SVC(random_state=SEED))]),
        {"clf__C": [0.1, 1, 10]},
    ),
    "RandomForest": (
        Pipeline([("scaler", StandardScaler()),
                  ("clf", RandomForestClassifier(random_state=SEED))]),
        {"clf__n_estimators": [50, 100]},
    ),
}

# %% Bước 7: chạy CV, lấy F-score
scoring = {"f1_macro": "f1_macro", "f1_weighted": "f1_weighted"}
rows = []
for name, (pipe, grid) in models.items():
    search = GridSearchCV(pipe, grid, cv=inner_cv, scoring="f1_macro", n_jobs=-1)
    res = cross_validate(search, X, y, cv=outer_cv, scoring=scoring,
                         return_train_score=True)
    rows.append({
        "model": name,
        "f1_macro_test": f"{res['test_f1_macro'].mean():.4f} ± {res['test_f1_macro'].std():.4f}",
        "f1_weighted_test": f"{res['test_f1_weighted'].mean():.4f} ± {res['test_f1_weighted'].std():.4f}",
        "f1_macro_train": f"{res['train_f1_macro'].mean():.4f}",
        "grid": str(grid),
    })
table = pd.DataFrame(rows)
print(table.to_string(index=False))

# %% Lưu kết quả vào results/
os.makedirs("results", exist_ok=True)
table["date"] = datetime.date.today().isoformat()
table["k_outer"], table["k_inner"], table["seed"] = K_OUTER, K_INNER, SEED
table["sklearn"] = sklearn.__version__
table.to_csv("results/m05_f1_cv.csv", index=False)

(195, 23)
status
1    147
0     48
Name: count, dtype: int64
       model   f1_macro_test f1_weighted_test f1_macro_train                             grid
         KNN 0.8757 ± 0.0742  0.9073 ± 0.0578         0.9647  {'clf__n_neighbors': [3, 5, 7]}
         SVM 0.8904 ± 0.0804  0.9199 ± 0.0603         0.9787         {'clf__C': [0.1, 1, 10]}
RandomForest 0.8737 ± 0.1171  0.9103 ± 0.0824         1.0000 {'clf__n_estimators': [50, 100]}
